In [1]:
import pandas as pd
from pathlib import Path

processed_path = Path("../data")

files = sorted(processed_path.glob("*"))

print("Available files:\n")

for file in files:
    print(file.name)

Available files:

assets.csv
costs.csv
failures.csv
maintenance.csv
sensor_readings.csv
sites.csv
work_orders.csv


In [2]:
PROJECT_ROOT = Path.cwd().parents[1]

print("Project root:")
print(PROJECT_ROOT)

csv_files = sorted(PROJECT_ROOT.rglob("*.csv"))

print(f"CSV files found: {len(csv_files)}")
print("=" * 70)

for file in csv_files:
    print(file.relative_to(PROJECT_ROOT))


Project root:
d:\STUDY\Data_Science_Courses\PROJECTS\13.Intelligent Equipment Operations Hub
CSV files found: 7
Intelligent-Equipment-Operations-Hub\data\assets.csv
Intelligent-Equipment-Operations-Hub\data\costs.csv
Intelligent-Equipment-Operations-Hub\data\failures.csv
Intelligent-Equipment-Operations-Hub\data\maintenance.csv
Intelligent-Equipment-Operations-Hub\data\sensor_readings.csv
Intelligent-Equipment-Operations-Hub\data\sites.csv
Intelligent-Equipment-Operations-Hub\data\work_orders.csv


In [5]:
## Load tables

maintenance = pd.read_csv("D:/STUDY/Data_Science_Courses/PROJECTS/13.Intelligent Equipment Operations Hub/Intelligent-Equipment-Operations-Hub/data/maintenance.csv")
work_orders = pd.read_csv("D:/STUDY/Data_Science_Courses/PROJECTS/13.Intelligent Equipment Operations Hub/Intelligent-Equipment-Operations-Hub/data/work_orders.csv")

In [6]:
print("Maintenance columns:")
print(maintenance.columns.tolist())

print("\nWork Order columns:")
print(work_orders.columns.tolist())

Maintenance columns:
['Maintenance_ID', 'Asset_ID', 'Failure_ID', 'Maintenance_Date', 'Maintenance_Type', 'Maintenance_Reason', 'Component', 'Technician_ID', 'Duration_Hours', 'Planned_Flag', 'Maintenance_Status', 'Parts_Replaced', 'Notes']

Work Order columns:
['WorkOrder_ID', 'Asset_ID', 'Maintenance_ID', 'Failure_ID', 'Created_DateTime', 'Scheduled_DateTime', 'Completed_DateTime', 'WorkOrder_Type', 'Priority', 'Assigned_Team', 'WorkOrder_Status', 'Estimated_Hours', 'Actual_Hours', 'Description', 'SLA_Target_Hours', 'SLA_Breached_Flag']


In [7]:
maintenance_action = maintenance.merge(
    work_orders,
    on="Maintenance_ID",
    how="left",
    suffixes=("_maintenance", "_workorder")
)

In [8]:
print(maintenance_action.shape)
print(maintenance_action.head())
print(maintenance_action.columns.tolist())

(310, 28)
  Maintenance_ID Asset_ID_maintenance Failure_ID_maintenance  \
0       MNT00001              AST0001                    NaN   
1       MNT00002              AST0001                    NaN   
2       MNT00003              AST0002                    NaN   
3       MNT00004              AST0002                    NaN   
4       MNT00005              AST0003                    NaN   

                Maintenance_Date Maintenance_Type  \
0  2026-07-18 02:00:00.000000000       Inspection   
1  2026-06-14 18:00:00.000000000       Preventive   
2  2026-08-03 03:00:00.000000000       Preventive   
3  2026-06-01 21:00:00.000000000       Preventive   
4  2026-07-11 15:00:00.000000000       Preventive   

                 Maintenance_Reason  Component Technician_ID  Duration_Hours  \
0      Routine equipment inspection  Fan Blade       TECH010            2.59   
1  Scheduled preventive maintenance      Motor       TECH008            3.53   
2  Scheduled preventive maintenance    Winding

In [11]:
maintenance_action_import = pd.DataFrame()

maintenance_action_import["Action_ID"] = (
    maintenance_action["Maintenance_ID"]
)

maintenance_action_import["Action_Title"] = (
    maintenance_action["Maintenance_Type"].astype(str)
    + " - "
    + maintenance_action["Asset_ID_maintenance"].astype(str)
)

maintenance_action_import["Issue_ID"] = (
    maintenance_action["Failure_ID_maintenance"]
)

maintenance_action_import["Action_Type"] = (
    maintenance_action["Maintenance_Type"]
)

In [12]:
def get_action_status(row):
    if pd.notna(row.get("Completed_Date")):
        return "Completed"
    elif pd.notna(row.get("Started_Date")):
        return "In Progress"
    else:
        return "Planned"

maintenance_action_import["Action_Status"] = maintenance_action.apply(
    get_action_status,
    axis=1
)


maintenance_action_import["Scheduled_Date"] = maintenance_action.get("Scheduled_Date")
maintenance_action_import["Started_Date"] = maintenance_action.get("Started_Date")
maintenance_action_import["Completed_Date"] = maintenance_action.get("Completed_Date")

maintenance_action_import["Notes"] = maintenance_action.get("Description")
maintenance_action_import["Estimated_Cost"] = maintenance_action.get("Estimated_Cost")
maintenance_action_import["Actual_Cost"] = maintenance_action.get("Actual_Cost")

In [ ]:
""" maintenance_action_import.to_csv(
    "maintenance_action_import.csv",
    index=False
)

print("maintenance_action_import.csv created successfully") """

maintenance_action_import.csv created successfully


In [14]:
print(maintenance_action_import.columns.tolist())

['Action_ID', 'Action_Title', 'Issue_ID', 'Action_Type', 'Action_Status', 'Scheduled_Date', 'Started_Date', 'Completed_Date', 'Notes', 'Estimated_Cost', 'Actual_Cost']


In [ ]:
## Manp Site_ID for Dateverse import for 'Assest' table


assets = pd.read_csv("D:/STUDY/Data_Science_Courses/PROJECTS/13.Intelligent Equipment Operations Hub/Intelligent-Equipment-Operations-Hub/data/assets.csv")

site_map = {
    "SITE001": "26d274a5-78b4-f111-aaad-6045bd0d0e15",
    "SITE002": "PUT_SITE002_GUID_HERE",
    "SITE003": "PUT_SITE003_GUID_HERE"
}

assets["Site_GUID"] = assets["Site_ID"].map(site_map)

print(assets[["Asset_ID", "Site_ID", "Site_GUID"]].head())
print("\nMissing Site GUIDs:", assets["Site_GUID"].isna().sum())

## assets.to_csv("assets_dataverse_import.csv", index=False)

  Asset_ID  Site_ID                             Site_GUID
0  AST0001  SITE001  26d274a5-78b4-f111-aaad-6045bd0d0e15
1  AST0002  SITE001  26d274a5-78b4-f111-aaad-6045bd0d0e15
2  AST0003  SITE001  26d274a5-78b4-f111-aaad-6045bd0d0e15
3  AST0004  SITE001  26d274a5-78b4-f111-aaad-6045bd0d0e15
4  AST0005  SITE001  26d274a5-78b4-f111-aaad-6045bd0d0e15

Missing Site GUIDs: 0
